# Unwrapping AI | Zero to Transformer
## Step 12 — Transformers: From Attention to a Tiny GPT
**Creator:** Nitin K. Sethi | **Google Colab ready**

Build intuition and a small decoder-only Transformer through four practical examples. You will visualize token embeddings, self-attention, causal masking, training loss, generated text, and learned attention. This is a teaching model, not a production LLM.

## Roadmap connection
Step 11 used backpropagation to train a neural network. Transformers also use backpropagation and gradient descent, but self-attention lets each token gather information from other tokens.

**Text → Tokens → Embeddings + Positions → Masked Self-Attention → Feed-forward network → Next-token prediction**

## Setup

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:",torch.__version__," | Device:",device)

# Example 1 — Tokenization, embeddings and position
Text is converted into token IDs. An embedding maps each ID to a vector. Positional information helps the model distinguish token order.

In [ ]:
sample="the cat sat"
vocab=sorted(set(sample)); stoi={c:i for i,c in enumerate(vocab)}; itos={i:c for c,i in stoi.items()}
ids=[stoi[c] for c in sample]
print("Text:",sample,"\nVocabulary:",vocab,"\nToken IDs:",ids)
emb=nn.Embedding(len(vocab),8)(torch.tensor(ids))
plt.figure(figsize=(9,3)); plt.imshow(emb.detach().numpy(),aspect="auto",cmap="viridis"); plt.colorbar(label="Value")
plt.yticks(range(len(sample)),list(sample)); plt.xlabel("Embedding dimension"); plt.ylabel("Character position"); plt.title("Example 1A — token embedding lookup"); plt.show()

In [ ]:
def sinusoidal_positions(length,d_model):
    pos=np.arange(length)[:,None]; dim=np.arange(d_model)[None,:]
    angle=pos/np.power(10000,(2*(dim//2))/d_model)
    pe=np.zeros((length,d_model)); pe[:,0::2]=np.sin(angle[:,0::2]); pe[:,1::2]=np.cos(angle[:,1::2])
    return pe
pe=sinusoidal_positions(40,32)
plt.figure(figsize=(10,4)); plt.imshow(pe,aspect="auto",cmap="coolwarm"); plt.colorbar(); plt.xlabel("Embedding dimension"); plt.ylabel("Position"); plt.title("Example 1B — sinusoidal positional encoding"); plt.show()

**Observe:** Each row is a position-specific signal. In a Transformer, positional information is combined with token embeddings.

# Example 2 — Scaled dot-product self-attention
Queries (Q) ask what information a token needs; keys (K) describe what each token offers; values (V) carry the information.

`Attention(Q,K,V) = softmax(QKᵀ / √dₖ) V`

In [ ]:
tokens=["The","cat","sat","there"]; x=torch.randn(1,len(tokens),8)
q=nn.Linear(8,8,bias=False)(x); k=nn.Linear(8,8,bias=False)(x); v=nn.Linear(8,8,bias=False)(x)
scores=q@k.transpose(-2,-1)/math.sqrt(8); weights=torch.softmax(scores,dim=-1); context=weights@v
print("Q/K/V:",q.shape,k.shape,v.shape," Context:",context.shape)
print("Row sums:",weights[0].sum(-1).detach().numpy().round(3))
plt.figure(figsize=(6,5)); plt.imshow(weights[0].detach().numpy(),cmap="Blues",vmin=0,vmax=1); plt.colorbar(label="Attention weight")
plt.xticks(range(4),tokens); plt.yticks(range(4),tokens); plt.xlabel("Key token"); plt.ylabel("Query token"); plt.title("Example 2 — self-attention heatmap")
for i in range(4):
 for j in range(4): plt.text(j,i,f"{weights[0,i,j].item():.2f}",ha="center",va="center")
plt.show()

**Interpretation:** A row shows how one query token distributes attention across key tokens. These weights are random at initialization, not learned language relationships.

# Example 3 — Causal masking
A decoder predicts the next token. It must not look at future tokens during training. A causal mask blocks positions above the diagonal before softmax.

In [ ]:
T=6; allowed=torch.tril(torch.ones(T,T)); raw=torch.randn(T,T)
masked=raw.masked_fill(allowed==0,float("-inf")); causal=torch.softmax(masked,dim=-1)
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].imshow(allowed,cmap="Greens",vmin=0,vmax=1); ax[0].set_title("Allowed positions")
ax[1].imshow(causal.numpy(),cmap="Blues",vmin=0,vmax=1); ax[1].set_title("Attention after causal mask")
for a in ax: a.set_xlabel("Key position"); a.set_ylabel("Query position")
plt.tight_layout(); plt.show()
print("Maximum future attention:",torch.triu(causal,diagonal=1).abs().max().item())
print("Row sums:",causal.sum(-1).numpy().round(3))

**Check:** Future attention is zero. Token position 0 can only see itself; later positions can see themselves and earlier positions.

# Example 4 — Train a tiny GPT-style character language model
Now combine token and position embeddings, masked self-attention, residual connections, a feed-forward network and a next-character prediction head. The small repetitive corpus makes the full training loop practical on Colab.

In [ ]:
corpus=("the little robot learns to read. the robot sees the moon and the stars. "
        "the moon shines at night. a small bird sings in the morning. "
        "the bird flies over the green garden. the robot writes a story about the bird. ")*80
chars=sorted(set(corpus)); stoi={c:i for i,c in enumerate(chars)}; itos={i:c for c,i in stoi.items()}
data=torch.tensor([stoi[c] for c in corpus],dtype=torch.long)
split=int(.9*len(data)); train_data,val_data=data[:split],data[split:]
B,block_size=32,64
def get_batch(which):
 src=train_data if which=="train" else val_data
 ix=torch.randint(len(src)-block_size-1,(B,))
 x=torch.stack([src[i:i+block_size] for i in ix]); y=torch.stack([src[i+1:i+block_size+1] for i in ix])
 return x.to(device),y.to(device)
print("Corpus length:",len(data)," Vocabulary:",len(chars)," Train/validation:",len(train_data),len(val_data))

## Model architecture
A compact one-block decoder: learned token and position embeddings → causal self-attention → residual + layer norm → feed-forward network → vocabulary logits.

In [ ]:
class CausalAttention(nn.Module):
 def __init__(self,d,block):
  super().__init__(); self.q=nn.Linear(d,d,bias=False); self.k=nn.Linear(d,d,bias=False); self.v=nn.Linear(d,d,bias=False)
  self.register_buffer("mask",torch.tril(torch.ones(block,block)))
 def forward(self,x):
  B,T,D=x.shape; q,k,v=self.q(x),self.k(x),self.v(x)
  score=q@k.transpose(-2,-1)/math.sqrt(D); score=score.masked_fill(self.mask[:T,:T]==0,float("-inf"))
  a=torch.softmax(score,dim=-1); return a@v,a
class TinyGPT(nn.Module):
 def __init__(self,vocab_size,d=48,block=64):
  super().__init__(); self.block=block; self.tok=nn.Embedding(vocab_size,d); self.pos=nn.Embedding(block,d)
  self.attn=CausalAttention(d,block); self.ln1=nn.LayerNorm(d)
  self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d)); self.ln2=nn.LayerNorm(d); self.head=nn.Linear(d,vocab_size)
 def forward(self,idx,targets=None):
  B,T=idx.shape; p=torch.arange(T,device=idx.device)
  x=self.tok(idx)+self.pos(p)[None,:,:]; a,att=self.attn(x); x=self.ln1(x+a); x=self.ln2(x+self.ff(x)); logits=self.head(x)
  loss=F.cross_entropy(logits.reshape(B*T,-1),targets.reshape(B*T)) if targets is not None else None
  return logits,loss,att
model=TinyGPT(len(chars),d=48,block=block_size).to(device)
print("Parameters:",sum(p.numel() for p in model.parameters()))

## Train and track training versus validation loss

In [ ]:
@torch.no_grad()
def evaluate():
 model.eval(); result={}
 for name in ["train","val"]:
  vals=[]
  for _ in range(8):
   x,y=get_batch(name); _,loss,_=model(x,y); vals.append(loss.item())
  result[name]=float(np.mean(vals))
 model.train(); return result
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-3)
steps=1000; interval=100; xs=[]; tr=[]; va=[]
for step in range(steps+1):
 if step%interval==0:
  r=evaluate(); xs.append(step); tr.append(r["train"]); va.append(r["val"]); print(f"step {step:4d} | train {r['train']:.3f} | val {r['val']:.3f}")
 x,y=get_batch("train"); _,loss,_=model(x,y); optimizer.zero_grad(set_to_none=True); loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
plt.figure(figsize=(8,4)); plt.plot(xs,tr,marker="o",label="Train"); plt.plot(xs,va,marker="o",label="Validation")
plt.xlabel("Training step"); plt.ylabel("Cross-entropy loss"); plt.title("Example 4A — tiny Transformer learning curve"); plt.legend(); plt.grid(alpha=.25); plt.show()

## Generate text from the trained model

In [ ]:
@torch.no_grad()
def generate(prompt,max_new=220,temperature=.8):
 model.eval(); idx=torch.tensor([[stoi[c] for c in prompt]],device=device)
 for _ in range(max_new):
  logits,_,_=model(idx[:,-model.block:]); probs=torch.softmax(logits[:,-1,:]/temperature,dim=-1)
  nxt=torch.multinomial(probs,1); idx=torch.cat([idx,nxt],dim=1)
 model.train(); return "".join(itos[i] for i in idx[0].tolist())
print(generate("the robot",220))
print("\nSecond prompt:")
print(generate("the bird",180))

## Visualize attention learned by the tiny model

In [ ]:
prompt="the robot sees"; idx=torch.tensor([[stoi[c] for c in prompt]],device=device)
with torch.no_grad(): _,_,att=model(idx)
a=att[0].cpu().numpy()
plt.figure(figsize=(7,5)); plt.imshow(a,cmap="magma",vmin=0,vmax=1); plt.colorbar(label="Attention weight")
plt.xticks(range(len(prompt)),list(prompt)); plt.yticks(range(len(prompt)),list(prompt)); plt.xlabel("Key position"); plt.ylabel("Query position"); plt.title("Example 4B — trained causal attention"); plt.show()

## Four experiments to try
1. **Tokenization and positions:** change `sample` and compare the embedding/position plots.
2. **Self-attention:** change the four tokens and inspect the heatmap.
3. **Causal mask:** change sequence length and confirm future attention remains zero.
4. **Tiny GPT:** change `steps`, `d` (embedding size), learning rate, corpus and prompts. Compare training/validation loss and generated text.

### Review questions
- Why are positional embeddings needed?
- What do Q, K and V do?
- Why does a decoder use a causal mask?
- What does rising validation loss alongside falling training loss suggest?
- Why is this tiny character model not equivalent to ChatGPT?

### Completion checklist
- [ ] Run all cells in Google Colab.
- [ ] Inspect the four examples and their visualizations.
- [ ] Confirm future attention is zero.
- [ ] Review training and validation loss.
- [ ] Generate text from two prompts.
- [ ] Save notebook and outputs to GitHub.

**This completes the 12-step Zero to Transformer learning roadmap.** This is a compact educational decoder-only model; production Transformers typically use multiple layers, multiple heads, larger datasets and more sophisticated training.